In [11]:
import pandas as pd

# Load the Excel file
file_path = 'Kappa002.xlsx'
data = pd.read_excel(file_path)

# Display the first few rows of the dataframe to understand its structure
data.head()

,6,6.1
0,6,6
1,8,8
2,4,4
3,6,6
4,8,8


In [14]:
# Extract the two columns to compare from the new data
# coder1 = data.iloc[:, 1].astype(str)
# coder2 = data.iloc[:, 2].astype(str)

# second run
coder1 = data.iloc[:, 0].astype(str)
coder2 = data.iloc[:, 1].astype(str)

# Third Run
# coder1 = data.iloc[:, 3].astype(str)
# coder2 = data.iloc[:, 4].astype(str)

from sklearn.metrics import cohen_kappa_score
# Recalculate Cohen's Kappa Score for the new data
kappa_score = cohen_kappa_score(coder1, coder2)
kappa_score

np.float64(0.9941688590001732)

In [15]:

import numpy as np
from scipy import stats
def kappa_variance(y1, y2):
    """
    Calculate the variance of Cohen's kappa statistic.
    """
    # Calculate the confusion matrix
    classes = np.unique(np.concatenate((y1, y2)))
    n_classes = len(classes)
    confusion = np.zeros((n_classes, n_classes), dtype=int)
    for i, class1 in enumerate(classes):
        for j, class2 in enumerate(classes):
            confusion[i, j] = np.sum((y1 == class1) & (y2 == class2))

    # Calculate marginal totals and total observations
    n = np.sum(confusion)
    pa = np.trace(confusion) / n
    pe_rows = np.sum(confusion, axis=0) / n
    pe_cols = np.sum(confusion, axis=1) / n
    pe = np.sum(pe_rows * pe_cols)

    # Calculate kappa variance
    var_kappa = (pa * (1 - pa)) / (n * (1 - pe)**2)
    return var_kappa

# Calculate variance of Cohen's kappa
variance_kappa = kappa_variance(coder1, coder2)
z_score = kappa_score / np.sqrt(variance_kappa)
p_value = 2 * (1 - stats.norm.cdf(np.abs(z_score)))

variance_kappa, z_score, p_value

(np.float64(2.4507884596112724e-08),
 np.float64(6350.490080728727),
 np.float64(0.0))

In [16]:
import numpy as np
from sklearn.metrics import cohen_kappa_score
from scipy import stats

def kappa_variance(y1, y2):
    """
    Calculate the variance of Cohen's kappa statistic.
    """
    # Calculate the confusion matrix
    classes = np.unique(np.concatenate((y1, y2)))
    n_classes = len(classes)
    confusion = np.zeros((n_classes, n_classes), dtype=int)
    for i, class1 in enumerate(classes):
        for j, class2 in enumerate(classes):
            confusion[i, j] = np.sum((y1 == class1) & (y2 == class2))

    # Calculate marginal totals and total observations
    n = np.sum(confusion)
    pa = np.trace(confusion) / n
    pe_rows = np.sum(confusion, axis=0) / n
    pe_cols = np.sum(confusion, axis=1) / n
    pe = np.sum(pe_rows * pe_cols)

    # Calculate kappa variance
    var_kappa = (pa * (1 - pa)) / (n * (1 - pe)**2)
    return var_kappa

# Calculate kappa score using sklearn.metrics
kappa_score = cohen_kappa_score(coder1, coder2)

# Calculate variance of Cohen's kappa
variance_kappa = kappa_variance(coder1, coder2)
z_score = kappa_score / np.sqrt(variance_kappa)
p_value = 2 * (1 - stats.norm.cdf(np.abs(z_score)))

# Print the results formatted to three decimal places
print(f"Variance of Kappa: {variance_kappa:.3f}")
print(f"Z-score: {z_score:.3f}")
print(f"P-value: {p_value:.5f}")

Variance of Kappa: 0.000
Z-score: 6350.490
P-value: 0.00000


In [ ]:
import numpy as np

def kappa_confidence_interval(column1, column2, alpha=0.05):
    """
    Compute the Cohen's Kappa score along with its confidence interval using bootstrapping.

    Parameters:
        column1, column2 (array-like): Columns to compute Cohen's Kappa score.
        alpha (float): Significance level for confidence interval (default is 0.05).

    Returns:
        kappa (float): Cohen's Kappa score.
        lower (float): Lower bound of confidence interval.
        upper (float): Upper bound of confidence interval.
        z (float): Z-score for the kappa.
        p_value (float): P-value for the kappa.
    """
    # Compute initial kappa score
    kappa = cohen_kappa_score(coder1, coder2)

    # Bootstrapping to calculate the confidence interval
    n = len(column1)
    bootstrap_samples = 10000
    kappa_samples = []

    for _ in range(bootstrap_samples):
        # Generate random indices with replacement
        indices = np.random.randint(0, n, n)
        bootstrap_kappa = cohen_kappa_score(column1[indices], column2[indices])
        kappa_samples.append(bootstrap_kappa)

    kappa_samples = np.array(kappa_samples)
    conf_interval = np.percentile(kappa_samples, [100 * alpha / 2, 100 * (1 - alpha / 2)])

    # Calculate the standard error and Z-score
    kappa_std = kappa_samples.std()
    z = kappa / kappa_std

    # Calculate p-value from Z-score
    from scipy.stats import norm
    p_value = norm.sf(abs(z)) * 2  # Two-tailed test

    return kappa, conf_interval[0], conf_interval[1], z, p_value

# Calculate Cohen's Kappa with confidence interval and p-value
kappa_stat = kappa_confidence_interval(coder1.values, coder2.values)
kappa_stat

In [ ]:
import numpy as np
from sklearn.metrics import cohen_kappa_score
from scipy.stats import norm

def kappa_confidence_interval(column1, column2, alpha=0.05):
    """
    Compute the Cohen's Kappa score along with its confidence interval using bootstrapping.

    Parameters:
        column1, column2 (array-like): Columns to compute Cohen's Kappa score.
        alpha (float): Significance level for confidence interval (default is 0.05).

    Returns:
        kappa (float): Cohen's Kappa score.
        lower (float): Lower bound of confidence interval.
        upper (float): Upper bound of confidence interval.
        z (float): Z-score for the kappa.
        p_value (float): P-value for the kappa.
    """
    # Compute initial kappa score
    kappa = cohen_kappa_score(column1, column2)

    # Bootstrapping to calculate the confidence interval
    n = len(column1)
    bootstrap_samples = 10000
    kappa_samples = []

    for _ in range(bootstrap_samples):
        # Generate random indices with replacement
        indices = np.random.randint(0, n, size=n)
        bootstrap_kappa = cohen_kappa_score(column1[indices], column2[indices])
        kappa_samples.append(bootstrap_kappa)

    kappa_samples = np.array(kappa_samples)
    conf_interval = np.percentile(kappa_samples, [100 * alpha / 2, 100 * (1 - alpha / 2)])

    # Calculate the standard error and Z-score
    kappa_std = kappa_samples.std()
    z = kappa / kappa_std

    # Calculate p-value from Z-score
    p_value = 2 * norm.sf(abs(z))  # Two-tailed test

    return kappa, conf_interval[0], conf_interval[1], z, p_value

kappa_stat = kappa_confidence_interval(coder1, coder2)
print(f"Cohen's Kappa: {kappa_stat[0]:.5f}, CI Lower: {kappa_stat[1]:.5f}, CI Upper: {kappa_stat[2]:.5f}, Z-score: {kappa_stat[3]:.5f}, P-value: {kappa_stat[4]:.5f}")

Cohen's Kappa: 0.59412, CI Lower: 0.54054, CI Upper: 0.64355, Z-score: 22.47130, P-value: 0.00000
